In [1]:
from llm_graph.core.nodes import FunctionalNode, ConditionalNode
from llm_graph.core.runner import GraphRunner
from llm_graph.llm.llm_call import LLMCall
from llm_graph.llm.response_functions import dummy_llm_response_fn, OpenAI_response_fn


In [2]:
llm = LLMCall(response_fn=dummy_llm_response_fn)

In [3]:
llm({'user_query':"Hello ThEre"})

{'raw_output': 'hello there',
 'input': 'Hello ThEre',
 'message_history': [{'role': 'user', 'content': 'Hello ThEre'},
  {'role': 'assistant', 'content': 'hello there'}]}

In [4]:
def conditional(state):
    cond = state.get('raw_output', '')
    if len(cond) % 2 == 0:
        return "even"
    return "odd"

In [5]:
def odd_func(state):
    return state | {"result" : "odd"} 

def even_func(state):
    return state | {"result": "even"}

In [6]:
node1 = FunctionalNode(func=llm, name="start", next_node_name="conditional")
node2 = ConditionalNode(condition_fn=conditional, name="conditional")
node3 = FunctionalNode(func=odd_func, name="odd")
node4 = FunctionalNode(func=even_func, name="even")

In [7]:
runner = GraphRunner(nodes = [node1, node2, node3, node4], start_node = "start")

In [8]:
runner.execute({"user_query": "hi"})

{'user_query': 'hi',
 'raw_output': 'hi',
 'input': 'hi',
 'message_history': [{'role': 'user', 'content': 'hi'},
  {'role': 'assistant', 'content': 'hi'}],
 'result': 'even'}

In [9]:
runner.execute({"user_query": "hello"})

{'user_query': 'hello',
 'raw_output': 'user : hi\nassistant : hi\nhello',
 'input': 'user : hi\nassistant : hi\nhello',
 'message_history': [{'role': 'user', 'content': 'hi'},
  {'role': 'assistant', 'content': 'hi'},
  {'role': 'user', 'content': 'hello'},
  {'role': 'assistant', 'content': 'user : hi\nassistant : hi\nhello'}],
 'result': 'even'}

In [10]:
runner.execute({"user_query": "hi"})

{'user_query': 'hi',
 'raw_output': 'user : hi\nassistant : hi\nuser : hello\nassistant : user : hi\nassistant : hi\nhello\nhi',
 'input': 'user : hi\nassistant : hi\nuser : hello\nassistant : user : hi\nassistant : hi\nhello\nhi',
 'message_history': [{'role': 'user', 'content': 'hi'},
  {'role': 'assistant', 'content': 'hi'},
  {'role': 'user', 'content': 'hello'},
  {'role': 'assistant', 'content': 'user : hi\nassistant : hi\nhello'},
  {'role': 'user', 'content': 'hi'},
  {'role': 'assistant',
   'content': 'user : hi\nassistant : hi\nuser : hello\nassistant : user : hi\nassistant : hi\nhello\nhi'}],
 'result': 'odd'}

In [11]:
runner.clear_message_history()

In [12]:
runner.execute({"user_input": "hi"})

{'user_query': 'hi',
 'raw_output': 'hi',
 'input': 'hi',
 'message_history': [{'role': 'user', 'content': 'hi'},
  {'role': 'assistant', 'content': 'hi'}],
 'result': 'even',
 'user_input': 'hi'}

In [13]:
runner.state_dict["message_history"]

[{'role': 'user', 'content': 'hi'}, {'role': 'assistant', 'content': 'hi'}]

In [14]:
runner.clear_message_history()

In [15]:
runner.state_dict["message_history"]

[]